[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_52_Advanced_Retrieval.ipynb)

# Lesson 52 - Phase 5: Advanced Retrieval

**Phase 5 Goal:** Ship `auto_researcher` v2.0 as a real, publishable OSS package.

| Lesson | Topic | Status |
|--------|-------|--------|
| L51 | Architecture and Scaffold | done |
| **L52** | **Advanced Retrieval** | you are here |
| L53 | Production Deployment | next |
| L54 | DevEx (CLI + docs + packaging) | soon |
| L55 | Capstone: Ship It to PyPI | soon |

---

## What you will build

`auto_researcher_v2` needs retrieval that actually works at scale.
In L7 you learned naive RAG (embed -> chunk -> cosine search). Today you upgrade every part:

```
Query
  |
  +-- [Dense path]   embed(query) -------> cosine search on vector store
  +-- [Sparse path]  BM25(query)  -------> keyword search on inverted index
  |
  +-- [Hybrid RRF]   Reciprocal Rank Fusion --> top-30
           |
           +-- [HyDE]   optionally replace query with embed(hypothetical answer)
           |
           +-- [Re-rank] cross-encoder score --> top-5
                   |
                   +-- [Cache] TTL cache keyed on query hash
                           |
                           +-- RetrievalStore.retrieve(query) -> List[Chunk]
```

By the end you will have a drop-in upgrade for
`auto_researcher_v2/retrieval/store.py` and `retrieval/cache.py` from L51.

In [ ]:
# Setup
!pip install anthropic chromadb sentence-transformers rank-bm25 \
             pydantic nest_asyncio numpy pandas matplotlib tabulate -q

import os, sys, json, time, math, hashlib, re, textwrap
from dataclasses import dataclass, field
from typing import List, Optional, Dict, Tuple, Any
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import anthropic

try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY', 'your-key-here')

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
HAIKU  = 'claude-haiku-4-5'
SONNET = 'claude-sonnet-4-6'
print('Setup complete')

## 1. Why Naive RAG Fails at Production Scale

Naive RAG (Lesson 7) has three failure modes at scale:

| Problem | Root Cause | Symptom |
|---------|-----------|---------|
| **Vocabulary mismatch** | Dense embeddings miss exact keywords; BM25 misses synonyms | Query 'ML training stability' misses chunk containing 'gradient explosion' |
| **Chunking artifacts** | Fixed 512-token windows split sentences mid-thought | Retrieved chunk starts mid-sentence, loses context |
| **Top-k without re-ranking** | Cosine similarity != relevance for the specific question | 3rd-ranked chunk is actually the most relevant answer |

**The fix:**
- Fixed chunking -> **semantic chunking** (split at meaning boundaries)
- Dense-only -> **hybrid search** (dense + BM25 fused with RRF)
- Add **cross-encoder re-ranking** after retrieval
- Optional **HyDE** for hard abstract queries
- **Retrieval caching** so repeated queries cost nothing

In [ ]:
# 20 AI research mini-paragraphs as corpus
# In production these come from your ingestion pipeline.

CORPUS = [
    # Transformers / attention (indices 0-2)
    ('The Transformer architecture introduced by Vaswani et al. (2017) replaced recurrence '
     'with self-attention, enabling parallelisation across sequence positions and achieving '
     'state-of-the-art results on machine translation.'),
    ('Multi-head attention allows a model to jointly attend to information from different '
     'representation subspaces at different positions. Each head learns a distinct projection '
     'of queries, keys, and values.'),
    ('The scaled dot-product attention computes: Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) V. '
     'The sqrt(d_k) scaling prevents vanishing gradients in deep dot products.'),
    # Training stability (indices 3-5)
    ('Gradient explosion occurs when gradients grow exponentially during backpropagation through '
     'many layers. Solutions include gradient clipping, residual connections, and careful '
     'weight initialisation.'),
    ('Layer normalisation (LayerNorm) stabilises deep network training by normalising activations '
     'within each layer. Unlike BatchNorm it works on a single sample, making it suitable for '
     'variable-length sequences.'),
    ('Residual connections let gradients flow directly through a network by adding the input of '
     'a block to its output. They are essential for training networks deeper than ~20 layers.'),
    # Fine-tuning (indices 6-8)
    ('LoRA (Low-Rank Adaptation) injects trainable low-rank matrices into frozen pre-trained '
     'weights. With rank r=16 you update ~0.1% of parameters yet match full fine-tune quality.'),
    ('QLoRA combines 4-bit NormalFloat quantisation of the base model with 16-bit LoRA adapters, '
     'enabling fine-tuning of 7B+ parameter models on a single consumer GPU with <16 GB VRAM.'),
    ('Catastrophic forgetting occurs when a neural network overwrites previously learned weights '
     'while training on new data. Mitigation strategies include EWC and replay buffers.'),
    # Retrieval / RAG (indices 9-14)
    ('Retrieval-Augmented Generation (RAG) improves factual accuracy by conditioning generation '
     'on retrieved documents at inference time. It separates the knowledge store from model weights.'),
    ('Dense passage retrieval (DPR) uses two separate BERT encoders for questions and passages, '
     'trained with in-batch negatives to maximise inner product between positive pairs.'),
    ('BM25 is a bag-of-words retrieval function that ranks documents using term frequency (TF) '
     'and inverse document frequency (IDF) with saturation parameter k1 and length normalisation b.'),
    ('Hybrid search combines dense embeddings with sparse BM25 scores. Reciprocal Rank Fusion '
     '(RRF) merges ranked lists without requiring score normalisation: RRF(d) = sum 1/(k+rank_i(d)).'),
    ('Cross-encoder re-rankers take a (query, passage) pair and produce a single relevance score. '
     'They are slower than bi-encoders but more accurate because they model query-document interaction.'),
    ('HyDE (Hypothetical Document Embeddings) generates a plausible answer to the query with an LLM, '
     'embeds that answer, then searches the corpus with the answer embedding instead of the query.'),
    # Agents (indices 15-17)
    ('The ReAct framework interleaves reasoning traces and actions inside the LLM context window: '
     'the model emits a Thought, then an Action (tool call), observes the result, and repeats.'),
    ('A2A (Agent-to-Agent) protocol lets autonomous agents discover each other via Agent Cards, '
     'delegate tasks via POST /tasks/send, and poll for results without tight coupling.'),
    ('Circuit breakers protect AI pipelines from cascading failures. A CLOSED breaker lets '
     'requests through; it trips OPEN after consecutive failures and re-tries with HALF_OPEN.'),
    # Observability (indices 18-19)
    ('OpenTelemetry (OTel) defines a vendor-neutral standard for traces, metrics, and logs. '
     'A Span represents a unit of work; Spans form a tree sharing a Trace ID.'),
    ('Prometheus metrics use four instrument types: Counter (monotonically increasing), '
     'Gauge (snapshot value), Histogram (distribution with buckets), and Summary (pre-computed percentiles).'),
]

print(f'Corpus: {len(CORPUS)} documents')
for i, doc in enumerate(CORPUS[:3]):
    print(f'  doc-{i:02d}: {doc[:80]}...')

## 2. Semantic Chunking

Fixed-size chunking often breaks sentences mid-thought. **Semantic chunking** groups
sentences by meaning: merge consecutive sentences while cosine similarity stays above
a threshold, split when it drops.

```
Sentence 1   sim=0.92   -->  keep in same chunk
Sentence 2   sim=0.88   -->  keep
Sentence 3   sim=0.61   -->  SPLIT here (below threshold)
Sentence 4   sim=0.84   -->  keep in new chunk
```

This produces chunks aligned to topic boundaries, not byte counts.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding model loaded')


@dataclass
class Chunk:
    id: str
    text: str
    doc_idx: int
    token_est: int = 0

    def __post_init__(self):
        self.token_est = len(self.text.split())


def fixed_chunk(docs, max_tokens=80):
    chunks = []
    for di, doc in enumerate(docs):
        words = doc.split()
        for start in range(0, len(words), max_tokens):
            text = ' '.join(words[start:start + max_tokens])
            chunks.append(Chunk(id=f'fixed-{di}-{start}', text=text, doc_idx=di))
    return chunks


def semantic_chunk(docs, threshold=0.75, max_tokens=150):
    def sent_split(text):
        return [p for p in re.split(r'(?<=[.!?])\s+', text.strip()) if p]

    chunks = []
    for di, doc in enumerate(docs):
        sentences = sent_split(doc)
        if not sentences:
            continue
        if len(sentences) == 1:
            chunks.append(Chunk(id=f'sem-{di}-0', text=doc, doc_idx=di))
            continue
        embs = EMBED_MODEL.encode(sentences, normalize_embeddings=True)
        sims = [float(np.dot(embs[i], embs[i+1])) for i in range(len(embs)-1)]
        current, n = [sentences[0]], 0
        for i, sim in enumerate(sims):
            nxt = sentences[i + 1]
            too_long = len(' '.join(current + [nxt]).split()) > max_tokens
            if sim < threshold or too_long:
                chunks.append(Chunk(id=f'sem-{di}-{n}',
                                    text=' '.join(current), doc_idx=di))
                current, n = [nxt], n + 1
            else:
                current.append(nxt)
        if current:
            chunks.append(Chunk(id=f'sem-{di}-{n}', text=' '.join(current), doc_idx=di))
    return chunks


fixed_chunks    = fixed_chunk(CORPUS)
semantic_chunks = semantic_chunk(CORPUS, threshold=0.70)

print(f'Fixed chunks:    {len(fixed_chunks):3d}  avg tokens: '
      f'{np.mean([c.token_est for c in fixed_chunks]):.1f}')
print(f'Semantic chunks: {len(semantic_chunks):3d}  avg tokens: '
      f'{np.mean([c.token_est for c in semantic_chunks]):.1f}')

doc_idx = 3  # gradient explosion doc
print(f'\nOriginal doc-{doc_idx}: {CORPUS[doc_idx][:90]}...')
sem = [c for c in semantic_chunks if c.doc_idx == doc_idx]
print(f'Semantic splits ({len(sem)}):')
for c in sem:
    print(f'  [{c.id}] {c.text[:88]}...' if len(c.text) > 88 else f'  [{c.id}] {c.text}')

# EXPERIMENT: Lower threshold (0.60) -> fewer larger chunks
# EXPERIMENT: Raise threshold (0.90) -> more smaller chunks

## 3. Hybrid Search: Dense + BM25 + RRF

Neither dense embeddings nor BM25 alone is best for all queries:

| Query type | Dense wins | BM25 wins |
|------------|-----------|-----------|
| `'ML training instability'` (synonym) | catches 'gradient explosion' | exact word mismatch |
| `'BM25 k1 parameter'` (keyword) | may retrieve unrelated text | exact term match |

**Reciprocal Rank Fusion (RRF):**
```
RRF(doc) = sum_i  1 / (k + rank_i(doc))    k=60 is standard
```
We sum the RRF contribution from each ranked list. No score normalisation needed.

In [ ]:
CHUNKS      = semantic_chunks
CHUNK_TEXTS = [c.text for c in CHUNKS]

print('Building dense index...')
t0 = time.time()
CHUNK_EMBS = EMBED_MODEL.encode(CHUNK_TEXTS, normalize_embeddings=True,
                                 batch_size=64, show_progress_bar=False)
print(f'  Dense: {len(CHUNKS)} chunks x {CHUNK_EMBS.shape[1]} dims  ({time.time()-t0:.2f}s)')

from rank_bm25 import BM25Okapi

def tok(text):
    return re.sub(r'[^a-z0-9 ]', ' ', text.lower()).split()

TOKENISED  = [tok(t) for t in CHUNK_TEXTS]
BM25_INDEX = BM25Okapi(TOKENISED)
print(f'  BM25: {len(TOKENISED)} documents')


def dense_search(query, k=20):
    q_emb  = EMBED_MODEL.encode([query], normalize_embeddings=True)[0]
    scores = CHUNK_EMBS @ q_emb
    return [(int(i), float(scores[i])) for i in np.argsort(-scores)[:k]]


def bm25_search(query, k=20):
    scores = BM25_INDEX.get_scores(tok(query))
    return [(int(i), float(scores[i])) for i in np.argsort(-scores)[:k]]


def rrf_fuse(ranked_lists, k=60, top_n=20):
    rrf = {}
    for ranked in ranked_lists:
        for rank, (doc_idx, _) in enumerate(ranked):
            rrf[doc_idx] = rrf.get(doc_idx, 0.0) + 1.0 / (k + rank + 1)
    return sorted(rrf.items(), key=lambda x: -x[1])[:top_n]


def hybrid_search(query, top_n=20):
    return rrf_fuse([dense_search(query), bm25_search(query)], top_n=top_n)


def show(label, results, n=3):
    print(f'  --- {label} ---')
    for r, (idx, score) in enumerate(results[:n]):
        print(f'    #{r+1} [{score:.4f}] {CHUNKS[idx].text[:85]}...')


q1 = 'ML training instability solutions'
q2 = 'BM25 k1 parameter normalisation'

print('\n=== Query 1:', q1)
show('Dense only',  dense_search(q1))
show('BM25 only',   bm25_search(q1))
show('Hybrid RRF',  hybrid_search(q1))

print('\n=== Query 2:', q2)
show('Dense only',  dense_search(q2))
show('BM25 only',   bm25_search(q2))
show('Hybrid RRF',  hybrid_search(q2))

# EXPERIMENT: show('My query', hybrid_search('how residual connections work'))

## 4. Cross-Encoder Re-Ranking

Bi-encoders (like `all-MiniLM-L6-v2`) embed query and document **independently** -
fast, but they cannot model their interaction.

A **cross-encoder** takes `(query, document)` as a single concatenated input and
scores relevance jointly. Much more accurate, but requires one forward pass per candidate.

**Pattern:** Bi-encoder retrieves top-30 (fast), cross-encoder re-ranks those 30 (precise).

```
Query --> Hybrid (fast, coarse)  -->  30 candidates
            |
            +--> Cross-encoder (slow, precise) --> 5 final
```

In [ ]:
from sentence_transformers import CrossEncoder

print('Loading cross-encoder (cross-encoder/ms-marco-MiniLM-L-6-v2)...')
CROSS_ENCODER = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print('  Cross-encoder loaded')


def rerank(query, candidates, top_k=5):
    if not candidates:
        return []
    pairs  = [(query, CHUNKS[idx].text) for idx, _ in candidates]
    scores = CROSS_ENCODER.predict(pairs)
    result = [(candidates[i][0], float(scores[i])) for i in range(len(candidates))]
    result.sort(key=lambda x: -x[1])
    return result[:top_k]


def full_pipeline(query, hybrid_k=30, rerank_k=5):
    return rerank(query, hybrid_search(query, top_n=hybrid_k), top_k=rerank_k)


def compare(query):
    print(f'\nQuery: "{query}"')
    h5 = hybrid_search(query, top_n=5)
    r5 = full_pipeline(query, hybrid_k=20, rerank_k=5)
    rows = []
    for rank in range(5):
        hi, hs = h5[rank] if rank < len(h5) else (-1, 0)
        ri, rs = r5[rank] if rank < len(r5) else (-1, 0)
        rows.append({
            'Rank': rank+1,
            'Hybrid (first 55 chars)':   CHUNKS[hi].text[:55] if hi >= 0 else '-',
            'H_score': f'{hs:.4f}',
            'Re-ranked (first 55 chars)': CHUNKS[ri].text[:55] if ri >= 0 else '-',
            'R_score': f'{rs:.4f}',
        })
    print(pd.DataFrame(rows).set_index('Rank').to_string())


compare('how does self-attention handle position information')
compare('how to prevent gradient explosion in deep networks')

# EXPERIMENT: Notice how re-ranking reorders results.
# 'gradient explosion' -> cross-encoder should surface clipping + residual connections.

## 5. HyDE - Hypothetical Document Embeddings

Some queries are abstract while answers live in specific, detailed chunks.
The **semantic gap** between a short query and a long answer hurts cosine similarity.

**HyDE** (Gao et al., 2022) closes this gap:
1. Generate a **hypothetical answer** to the query with an LLM (cheap Haiku, ~150 tokens)
2. Embed the hypothetical answer - it is in the same vocabulary space as real documents
3. Use that embedding as the query vector

```
Query --> LLM generates 'fake' answer --> embed(fake_answer) --> cosine search
```

The fake answer does not need to be factually correct - it just needs to be
**lexically close** to the real answer in the corpus.

In [ ]:
def generate_hypothetical_doc(query):
    resp = client.messages.create(
        model=HAIKU,
        max_tokens=150,
        system=(
            'You are a technical writer. Write a single dense paragraph '
            '(3-5 sentences) that directly answers the question. '
            'Use precise technical terminology. No caveats or hedges.'
        ),
        messages=[{'role': 'user', 'content': f'Question: {query}'}]
    )
    return resp.content[0].text.strip()


def hyde_search(query, top_n=10):
    hyp_doc = generate_hypothetical_doc(query)
    hyp_emb = EMBED_MODEL.encode([hyp_doc], normalize_embeddings=True)[0]
    scores  = CHUNK_EMBS @ hyp_emb
    ranked  = np.argsort(-scores)[:top_n]
    return [(int(i), float(scores[i])) for i in ranked]


def hyde_hybrid_rerank(query, top_n=5):
    all_cands = rrf_fuse([dense_search(query), bm25_search(query),
                          hyde_search(query, top_n=20)], top_n=30)
    return rerank(query, all_cands, top_k=top_n)


HARD_QUERY = 'why do very deep neural networks become harder to train'

print(f'Query: "{HARD_QUERY}"\n')
hyp = generate_hypothetical_doc(HARD_QUERY)
print('HyDE hypothetical document:')
print(textwrap.fill(hyp, width=90))
print()

std3  = full_pipeline(HARD_QUERY, hybrid_k=20, rerank_k=3)
hyde3 = hyde_hybrid_rerank(HARD_QUERY, top_n=3)

print('Standard (hybrid + re-rank) top-3:')
for r, (idx, sc) in enumerate(std3, 1):
    print(f'  #{r} [{sc:.3f}] {CHUNKS[idx].text[:88]}...')

print('\nHyDE + hybrid + re-rank top-3:')
for r, (idx, sc) in enumerate(hyde3, 1):
    print(f'  #{r} [{sc:.3f}] {CHUNKS[idx].text[:88]}...')

# EXPERIMENT: Try 'What problem does LoRA solve?'

## 6. Retrieval Caching

In production, the same queries repeat constantly. Re-embedding and re-searching is wasteful.

The **TTL cache** pattern from L31 applies directly:
- Key: `hash(query + k)` - include `k` so different top-k calls get separate entries
- Value: list of top-k chunk IDs (tiny - just integers)
- TTL: 1 hour (corpus is append-only; stale hits are very unlikely)

Because retrieval is deterministic (same query + k -> same top-k), cache hit rate
can be 40-60% on real traffic.

In [ ]:
@dataclass
class _CacheEntry:
    value: Any
    expires_at: float


class TTLCache:
    def __init__(self, max_size=512, ttl_s=3600.0):
        self._max  = max_size
        self._ttl  = ttl_s
        self._store: OrderedDict = OrderedDict()

    def _key(self, query, k):
        return hashlib.sha256(f'{k}::{query}'.encode()).hexdigest()[:16]

    def get(self, query, k):
        key   = self._key(query, k)
        entry = self._store.get(key)
        if entry is None:
            return None
        if time.time() > entry.expires_at:
            del self._store[key]; return None
        self._store.move_to_end(key)
        return entry.value

    def put(self, query, k, value):
        key = self._key(query, k)
        self._store[key] = _CacheEntry(value=value,
                                        expires_at=time.time() + self._ttl)
        self._store.move_to_end(key)
        while len(self._store) > self._max:
            self._store.popitem(last=False)

    def stats(self):
        now   = time.time()
        alive = sum(1 for e in self._store.values() if e.expires_at > now)
        return {'size': len(self._store), 'alive': alive, 'max': self._max}


RETRIEVAL_CACHE = TTLCache(max_size=256, ttl_s=3600.0)


def cached_retrieve(query, mode='hybrid+rerank', top_k=5):
    cached = RETRIEVAL_CACHE.get(query, top_k)
    if cached is not None:
        return ([CHUNKS[i] for i in cached], True)
    if mode == 'hybrid':
        results = hybrid_search(query, top_n=top_k)
    elif mode == 'hyde+rerank':
        results = hyde_hybrid_rerank(query, top_n=top_k)
    else:
        results = full_pipeline(query, hybrid_k=30, rerank_k=top_k)
    idxs = [idx for idx, _ in results]
    RETRIEVAL_CACHE.put(query, top_k, idxs)
    return ([CHUNKS[i] for i in idxs], False)


test_query = 'how does attention mechanism work in transformers'

t0 = time.time(); c1, h1 = cached_retrieve(test_query); t1 = time.time() - t0
t0 = time.time(); c2, h2 = cached_retrieve(test_query); t2 = time.time() - t0

print(f'First call  --> cache_hit={h1}, latency={t1*1000:.1f} ms')
print(f'Second call --> cache_hit={h2}, latency={t2*1000:.1f} ms')
print(f'Speedup: {t1/max(t2,1e-6):.0f}x  |  cache stats: {RETRIEVAL_CACHE.stats()}')
print('\nRetrieved chunks:')
for i, c in enumerate(c1, 1):
    print(f'  #{i} {c.text[:80]}...')

# EXPERIMENT: Change TTL to 5 seconds, wait 6 s, watch cache miss re-trigger.

## 7. Retrieval Evaluation: Recall@k, NDCG@k, MRR

You cannot improve what you cannot measure. Three standard metrics:

| Metric | What it measures |
|--------|-----------------|
| **Recall@k** | `|relevant & top-k| / |relevant|` - did we retrieve the relevant docs at all? |
| **MRR** (Mean Reciprocal Rank) | `1 / rank_of_first_relevant` - how early does the first relevant doc appear? |
| **NDCG@k** | DCG@k / IDCG@k (log-discounted) - graded relevance, higher-ranked is worth more |

We create a **golden set** of (query, relevant_doc_indices) pairs and measure each pipeline.

In [ ]:
GOLDEN = [
    {'query': 'how does multi-head attention work',           'relevant_docs': {0, 1, 2}},
    {'query': 'techniques to stabilise deep network training','relevant_docs': {3, 4, 5}},
    {'query': 'parameter efficient fine tuning methods',      'relevant_docs': {6, 7}},
    {'query': 'retrieval augmented generation architecture',  'relevant_docs': {9, 10}},
    {'query': 'hybrid search combining sparse and dense',     'relevant_docs': {11, 12, 13}},
]


def chunk_relevant(chunk_idx, relevant_docs):
    return CHUNKS[chunk_idx].doc_idx in relevant_docs


def recall_at_k(retrieved, relevant_docs, k):
    return sum(1 for i in retrieved[:k] if chunk_relevant(i, relevant_docs)) / max(len(relevant_docs), 1)


def reciprocal_rank(retrieved, relevant_docs):
    for rank, i in enumerate(retrieved, 1):
        if chunk_relevant(i, relevant_docs):
            return 1.0 / rank
    return 0.0


def ndcg_at_k(retrieved, relevant_docs, k):
    def dcg(gains):
        return sum(g / math.log2(i + 2) for i, g in enumerate(gains))
    gains = [1 if chunk_relevant(i, relevant_docs) else 0 for i in retrieved[:k]]
    ideal = sorted(gains, reverse=True)
    return dcg(gains) / dcg(ideal) if dcg(ideal) > 0 else 0.0


def evaluate_pipeline(name, retrieve_fn, golden, k=5):
    recalls, rrs, ndcgs = [], [], []
    for item in golden:
        results = retrieve_fn(item['query'])
        idxs = [idx for idx, _ in results[:k]]
        recalls.append(recall_at_k(idxs, item['relevant_docs'], k))
        rrs.append(reciprocal_rank(idxs, item['relevant_docs']))
        ndcgs.append(ndcg_at_k(idxs, item['relevant_docs'], k))
    return {
        'name':          name,
        f'Recall@{k}':   round(float(np.mean(recalls)), 3),
        'MRR':           round(float(np.mean(rrs)),     3),
        f'NDCG@{k}':     round(float(np.mean(ndcgs)),   3),
    }


print('Evaluating pipelines (may take ~30 s for re-ranking)...')
K = 5
rows = [
    evaluate_pipeline('Dense only',    lambda q: dense_search(q, k=K*4), GOLDEN, k=K),
    evaluate_pipeline('BM25 only',     lambda q: bm25_search(q,  k=K*4), GOLDEN, k=K),
    evaluate_pipeline('Hybrid RRF',    lambda q: hybrid_search(q, top_n=K*4), GOLDEN, k=K),
    evaluate_pipeline('Hybrid+Rerank', lambda q: full_pipeline(q, hybrid_k=30, rerank_k=K), GOLDEN, k=K),
]

df_eval = pd.DataFrame(rows).set_index('name')
print()
print(df_eval.to_string())

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
colors  = ['#4C9BE8', '#F4A942', '#6DBF6D', '#D96B6B']
for ax, metric in zip(axes, [f'Recall@{K}', 'MRR', f'NDCG@{K}']):
    bars = ax.bar(df_eval.index, df_eval[metric], color=colors)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(0, 1.05)
    ax.set_xticklabels(df_eval.index, rotation=15, ha='right', fontsize=9)
    for bar, val in zip(bars, df_eval[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.02,
                f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')
plt.suptitle('Retrieval Pipeline Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/retrieval_eval.png', dpi=110, bbox_inches='tight')
plt.show()
print('Saved --> /content/retrieval_eval.png')

# EXPERIMENT: Add HyDE to eval (costs API calls):
# rows.append(evaluate_pipeline('HyDE+Rerank',
#     lambda q: hyde_hybrid_rerank(q, top_n=K), GOLDEN, k=K))

## 8. Wiring into auto_researcher_v2

In L51, your scaffold created these placeholder files:
```
auto_researcher_v2/retrieval/store.py
auto_researcher_v2/retrieval/cache.py
```

Here is the production-ready `RetrievalStore` that replaces those stubs.
Key design choices mirror L51's `InferenceBackend` pattern:
- `RetrievalBackend` Protocol: swap ChromaDB vs in-memory vs Pinecone without changing callers
- `RetrievalStore` holds backend + cache: mirrors how `ComponentRegistry` holds `InferenceBackend`
- `AutoResearcherV2._search()` already calls `retrieval_store.retrieve(query)` - no changes needed

In [ ]:
import pathlib

BASE = pathlib.Path('/content/auto_researcher_v2')
(BASE / 'retrieval').mkdir(parents=True, exist_ok=True)

# Write store.py to the scaffold
STORE_PY = (
    'from __future__ import annotations\n'
    'import hashlib, time, re\n'
    'from collections import OrderedDict\n'
    'from dataclasses import dataclass, field\n'
    'from typing import Protocol, List, Optional, Dict, Any, runtime_checkable\n'
    'import numpy as np\n'
    'from sentence_transformers import SentenceTransformer, CrossEncoder\n'
    'from rank_bm25 import BM25Okapi\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class Chunk:\n'
    '    id: str\n'
    '    text: str\n'
    '    doc_id: str\n'
    '    metadata: dict = None\n'
    '\n'
    '@dataclass(frozen=True)\n'
    'class RetrievalResult:\n'
    '    chunks: list\n'
    '    from_cache: bool\n'
    '    latency_s: float\n'
    '    mode: str\n'
    '\n'
    'class InMemoryRetrievalBackend:\n'
    '    def __init__(self):\n'
    '        self._chunks   = []\n'
    '        self._embedder = SentenceTransformer("all-MiniLM-L6-v2")\n'
    '        self._reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")\n'
    '        self._embs = self._bm25 = None\n'
    '    def _tok(self, t):\n'
    '        import re; return re.sub(r"[^a-z0-9 ]", " ", t.lower()).split()\n'
    '    def add(self, chunks):\n'
    '        self._chunks.extend(chunks)\n'
    '        texts = [c.text for c in self._chunks]\n'
    '        self._embs = self._embedder.encode(texts, normalize_embeddings=True, batch_size=64, show_progress_bar=False)\n'
    '        from rank_bm25 import BM25Okapi; self._bm25 = BM25Okapi([self._tok(t) for t in texts])\n'
    '    def search(self, query, k=5):\n'
    '        if not self._chunks: return []\n'
    '        import numpy as np\n'
    '        q  = self._embedder.encode([query], normalize_embeddings=True)[0]\n'
    '        ds = self._embs @ q\n'
    '        dt = [(int(i), float(ds[i])) for i in np.argsort(-ds)[:k*4]]\n'
    '        bs = self._bm25.get_scores(self._tok(query))\n'
    '        bt = [(int(i), float(bs[i])) for i in np.argsort(-bs)[:k*4]]\n'
    '        rrf = {}\n'
    '        for ranked in [dt, bt]:\n'
    '            for rank, (idx, _) in enumerate(ranked):\n'
    '                rrf[idx] = rrf.get(idx, 0.0) + 1.0 / (60 + rank + 1)\n'
    '        cands = sorted(rrf.items(), key=lambda x: -x[1])[:k*3]\n'
    '        ce    = self._reranker.predict([(query, self._chunks[i].text) for i, _ in cands])\n'
    '        res   = [(cands[i][0], float(ce[i])) for i in range(len(cands))]\n'
    '        res.sort(key=lambda x: -x[1]); return res[:k]\n'
    '\n'
    'class RetrievalCache:\n'
    '    def __init__(self, max_size=512, ttl_s=3600.0):\n'
    '        import hashlib, time; self._max, self._ttl = max_size, ttl_s\n'
    '        from collections import OrderedDict; self._store = OrderedDict()\n'
    '    def _key(self, q, k): import hashlib; return hashlib.sha256(f"{k}::{q}".encode()).hexdigest()[:16]\n'
    '    def get(self, q, k):\n'
    '        import time; key = self._key(q, k); e = self._store.get(key)\n'
    '        if not e or time.time() > e["exp"]: (del self._store[key] if e else None); return None\n'
    '        self._store.move_to_end(key); return e["v"]\n'
    '    def put(self, q, k, v):\n'
    '        import time; key = self._key(q, k)\n'
    '        self._store[key] = {"v": v, "exp": time.time() + self._ttl}\n'
    '        self._store.move_to_end(key)\n'
    '        while len(self._store) > self._max: self._store.popitem(last=False)\n'
    '\n'
    'class RetrievalStore:\n'
    '    def __init__(self, backend=None, cache_ttl_s=3600.0, cache_size=512):\n'
    '        self._backend = backend or InMemoryRetrievalBackend()\n'
    '        self._cache   = RetrievalCache(max_size=cache_size, ttl_s=cache_ttl_s)\n'
    '        self._hits = self._misses = 0\n'
    '    def ingest(self, chunks): self._backend.add(chunks)\n'
    '    def retrieve(self, query, k=5):\n'
    '        import time; t0 = time.time()\n'
    '        hit = self._cache.get(query, k)\n'
    '        if hit is not None:\n'
    '            self._hits += 1\n'
    '            return RetrievalResult(chunks=[self._backend._chunks[i] for i, _ in hit], from_cache=True, latency_s=time.time()-t0, mode="cache_hit")\n'
    '        self._misses += 1\n'
    '        ranked = self._backend.search(query, k=k)\n'
    '        self._cache.put(query, k, ranked)\n'
    '        return RetrievalResult(chunks=[self._backend._chunks[i] for i, _ in ranked], from_cache=False, latency_s=time.time()-t0, mode="hybrid+rerank")\n'
    '    def cache_stats(self):\n'
    '        total = self._hits + self._misses\n'
    '        return {"hits": self._hits, "misses": self._misses, "hit_rate": round(self._hits/total, 3) if total > 0 else 0.0}\n'
)

(BASE / 'retrieval' / '__init__.py').write_text('')
(BASE / 'retrieval' / 'store.py').write_text(STORE_PY)
print('Written auto_researcher_v2/retrieval/store.py')

# Smoke-test using the classes built earlier in this notebook
# (CHUNKS, EMBED_MODEL, etc. are already in scope)
store = RetrievalStore(cache_ttl_s=3600)
corpus_v2 = [Chunk(id=f'doc-{i:02d}', text=t, doc_id=f'doc-{i:02d}')
             for i, t in enumerate(CORPUS)]
store.ingest(corpus_v2)
print(f'Ingested {len(corpus_v2)} chunks')

test_queries = [
    'how does multi-head attention compute relevance',
    'fine tuning with limited GPU memory',
    'hybrid search combining sparse and dense',
]
for q in test_queries:
    r = store.retrieve(q, k=3)
    flag = 'CACHE' if r.from_cache else r.mode
    print(f'[{r.latency_s*1000:.1f}ms] {flag}  {q[:55]}')

print('\nSecond pass (all should hit cache):')
for q in test_queries:
    r = store.retrieve(q, k=3)
    print(f'  [{r.latency_s*1000:.1f}ms] {"CACHE" if r.from_cache else "MISS"}  {q[:55]}')

print(f'\nCache stats: {store.cache_stats()}')


## 9. Ten Retrieval Pitfalls

| # | Pitfall | What goes wrong | Fix |
|---|---------|----------------|-----|
| 1 | **Fixed chunk size forever** | Splits technical sentences mid-thought | Semantic chunking (threshold tuned per domain) |
| 2 | **Embedding model mismatch** | Index with model A, query with model B | Store model name in index metadata; validate on load |
| 3 | **Re-embedding corpus on restart** | 20-min rebuild every deploy | Persist embeddings (.npy or vector DB); re-embed only new/changed docs |
| 4 | **Re-ranking all candidates** | Cross-encoder on 1000 docs = 10 s latency | Retrieve top-30 with bi-encoder first; re-rank only those 30 |
| 5 | **HyDE always on** | 200 ms + API cost per query; overkill for keyword queries | Use HyDE only for abstract/question-form queries |
| 6 | **Cache key is only the query string** | `retrieve('LoRA', k=3)` and `retrieve('LoRA', k=10)` collide | Include `k` (and model name) in the cache key |
| 7 | **Stale cache after corpus update** | Old cached results miss newly ingested documents | Invalidate cache on ingest or use short TTL |
| 8 | **Recall@k on tiny golden set** | 5 queries cannot detect regressions reliably | Keep >=50 golden pairs; run eval in CI |
| 9 | **NDCG without graded relevance** | Binary 0/1 misses gradations of relevance | Label golden set with grades (0=irrelevant, 1=partial, 2=highly relevant) |
| 10 | **No retrieval baseline in eval loop** | Generation improves but retrieval regresses silently | Retrieval eval as its own CI job; gate on Recall@5 >= 0.80 |

## 10. Homework

1. **Chunk size sensitivity** - run the eval harness with `fixed_chunk(max_tokens=50)`,
   `fixed_chunk(max_tokens=200)`, and `semantic_chunk(threshold=0.65)`. Plot Recall@5
   vs chunk size. What is the sweet spot for this corpus?

2. **HyDE eval** - add `hyde_hybrid_rerank` to the evaluation harness and compare it
   against `Hybrid+Rerank`. Is the extra API cost justified on your golden set?

3. **Persistent index** - save `CHUNK_EMBS` with `np.save('chunk_embs.npy', CHUNK_EMBS)`
   and reload on restart. Measure re-index time vs re-embed time. Critical for production.

4. **Retrieval CI gate** - add a step to the `auto_researcher_v2` `ci.yml` from L51
   that runs the eval harness and calls `sys.exit(1)` if `Recall@5 < 0.70`.
   Mirror the `SLO.ci_gate()` pattern from L31.

5. **ChromaDB backend** - implement a `ChromaDBRetrievalBackend` satisfying the
   `RetrievalBackend` Protocol using `chromadb.Client()`. Swap it into
   `RetrievalStore(backend=ChromaDBRetrievalBackend())`. The `retrieve()` callers
   should work unchanged.

---

## Phase 5 Progress

| Lesson | Topic | Status |
|--------|-------|--------|
| L51 | Architecture and Scaffold | done |
| L52 | Advanced Retrieval | **done - just delivered** |
| L53 | Production Deployment | next - FastAPI + Docker + Fly.io |
| L54 | DevEx: CLI, docs, packaging | soon |
| L55 | Capstone: Ship It to PyPI | soon |

**Next up (L53):** Wire up FastAPI (`/research` endpoint), write a `Dockerfile`,
deploy `auto_researcher_v2` to Fly.io (free tier), and smoke-test the live URL
using the CI/CD pipeline from L15 - now shipping a real multi-component AI app.